<a href="https://colab.research.google.com/github/roserocarlos/StatAI-Basics/blob/main/Ejercicio4/Cultivating_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cultivating Machine Learning
### Predicción del estado de salud de un cultivo a partir de sensores IoT reales

Dataset: **Plant_health_data.csv** (Kaggle — [gowthamduggirala/plant-health-data](https://www.kaggle.com/datasets/gowthamduggirala/plant-health-data)), 1200 registros reales con 14 columnas de sensores agrícolas.

Objetivo: predecir `Plant_Health_Status` (Healthy / Moderate Stress / High Stress) a partir de 10 sensores numéricos. El notebook está organizado en 8 sesiones incrementales — el código de cada una continúa el de la anterior, y la Sesión 8 usa todo el pipeline acumulado.

## Paso 0 — Descargar el dataset desde Kaggle

Necesitas un token de la API de Kaggle (`kaggle.json`): en tu cuenta de Kaggle ve a *Settings → API → Create New Token*, descarga el archivo y súbelo cuando la celda lo pida.

In [ ]:
# Ejecutar una sola vez por sesión de Colab
!pip install -q kaggle

from google.colab import files
print("Sube tu archivo kaggle.json (Kaggle > Settings > API > Create New Token)")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d gowthamduggirala/plant-health-data -p ./data --unzip
!ls ./data

## Sesión 1 — Del dato al DataFrame

El dato rectangular (Data Frame): fila = registro, columna = feature/predictor. Cargamos el CSV y separamos los 10 sensores numéricos del target categórico `Plant_Health_Status`.

In [ ]:
# ==========================================
# SESION 1 - CARGA Y ESTRUCTURA DE DATOS
# ==========================================
import pandas as pd
import numpy as np

RUTA_CSV = "./data/Plant_health_data.csv"
df = pd.read_csv(RUTA_CSV, parse_dates=["Timestamp"])

print("--- SESION 1: Dimensiones del dataset ---")
print(f"Registros (filas): {df.shape[0]} | Caracteristicas (columnas): {df.shape[1]}\n")
print("--- Tipos de datos por sensor ---")
print(df.dtypes)

SENSORES_NUM = ["Soil_Moisture", "Ambient_Temperature", "Soil_Temperature",
                "Humidity", "Light_Intensity", "Soil_pH",
                "Nitrogen_Level", "Phosphorus_Level", "Potassium_Level",
                "Electrochemical_Signal"]
TARGET = "Plant_Health_Status"

**Ejercicio:** confirma cuántos registros y columnas tiene tu copia del dataset, y verifica que `SENSORES_NUM` coincide exactamente con los nombres de columnas que ves en `df.dtypes`.

## Sesión 2 — Estadística descriptiva y limpieza robusta

La mediana es un estimador robusto frente a outliers (típicos de fallos de sensores IoT). Imputamos cada sensor con su mediana y eliminamos solo los registros sin target.

In [ ]:
# ==========================================
# SESION 2 - ESTADISTICA Y LIMPIEZA DE NULOS
# ==========================================
print("--- SESION 2: Estadisticas Descriptivas Iniciales ---")
print(df[SENSORES_NUM].describe().T[["mean", "50%", "std", "min", "max"]])

print("\nValores nulos detectados por sensor antes de la limpieza:")
print(df[SENSORES_NUM].isnull().sum())

for variable in SENSORES_NUM:
    mediana_robusta = df[variable].median()
    df[variable] = df[variable].fillna(mediana_robusta)

df = df.dropna(subset=[TARGET])

print("\nValores nulos despues de la limpieza robusta:")
print(df[SENSORES_NUM].isnull().sum())

**Ejercicio:** compara la media y la mediana de `Light_Intensity`. ¿Cuál sensor tiene mayor diferencia entre ambas? ¿Qué te dice eso sobre la presencia de outliers en ese sensor?

## Sesión 3 — Análisis Exploratorio de Datos (EDA)

Filosofía de Tukey: "el modelo siempre debe seguir a los datos". Histograma de `Light_Intensity` y matriz de correlación de Pearson entre los 10 sensores.

In [ ]:
# ==========================================
# SESION 3 - ANALISIS EXPLORATORIO DE DATOS (EDA)
# ==========================================
import matplotlib.pyplot as plt

print("--- SESION 3: Distribuciones y Correlacion ---")

plt.figure(figsize=(6, 3))
df["Light_Intensity"].hist(bins=25, color="teal", edgecolor="black")
plt.title("Distribucion de Intensidad Luminica (lux)")
plt.xlabel("Light_Intensity"); plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

matriz_corr = df[SENSORES_NUM].corr()
print("Matriz de correlacion de Pearson (sensores numericos):")
print(matriz_corr.round(2))

**Ejercicio:** identifica el par de sensores con la correlación más alta (en valor absoluto, sin contar la diagonal) y el par con la correlación más cercana a cero. Interpreta ambos casos en el contexto agronómico del cultivo.

## Sesión 4 — Relaciones y transformación de escala de potencias (Tukey)

`Light_Intensity` tiene cola larga → aplicamos `log10(x + 1)` para linealizar su relación con `Chlorophyll_Content` y comparamos la correlación antes/después.

In [ ]:
# ==========================================
# SESION 4 - TRANSFORMACION Y LINEALIZACION (Tukey)
# ==========================================
print("--- SESION 4: Escala de Potencias sobre Light_Intensity ---")

plt.figure(figsize=(6, 3))
plt.scatter(df["Soil_Moisture"], df["Chlorophyll_Content"], alpha=0.5, color="tomato")
plt.title("Humedad de Suelo vs Contenido de Clorofila")
plt.xlabel("Soil_Moisture (%)"); plt.ylabel("Chlorophyll_Content")
plt.tight_layout()
plt.show()

df["Light_Intensity_log"] = np.log10(df["Light_Intensity"] + 1)

corr_original = df["Light_Intensity"].corr(df["Chlorophyll_Content"])
corr_transf = df["Light_Intensity_log"].corr(df["Chlorophyll_Content"])
print(f"Correlacion original Light vs Clorofila: {corr_original:.4f}")
print(f"Correlacion log(Light) vs Clorofila:      {corr_transf:.4f}")

**Ejercicio:** ¿por qué se usa `log10(x + 1)` y no `log10(x)` directamente? Prueba aplicar la transformación a otro sensor con distribución sesgada (por ejemplo `Potassium_Level`) y observa si mejora alguna correlación de interés.

## Sesión 5 — Partición y estandarización (Z-score)

Estandarizamos con `StandardScaler` ajustado **solo** en entrenamiento (evita fuga de datos) y particionamos 80/20 de forma estratificada.

In [ ]:
# ==========================================
# SESION 5 - PARTICION Y ESTANDARIZACION
# ==========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

print("--- SESION 5: Particion de Datos y Escalado Z-Score ---")

predictores = ["Soil_Moisture", "Ambient_Temperature", "Soil_Temperature",
               "Humidity", "Light_Intensity_log", "Soil_pH",
               "Nitrogen_Level", "Phosphorus_Level", "Potassium_Level",
               "Electrochemical_Signal"]

X = df[predictores]
le = LabelEncoder()
y = le.fit_transform(df[TARGET])   # Healthy / Moderate Stress / High Stress -> 0,1,2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Entrenamiento: {X_train_scaled.shape} | Prueba: {X_test_scaled.shape}")
print(f"Clases codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}")

**Ejercicio:** quita `stratify=y` de `train_test_split`, vuelve a correr la celda y compara cuántos registros de "High Stress" quedan en el set de prueba. ¿Por qué importa esto?

## Sesión 6 — Modelo lineal vs. Árbol (CART)

Regresión Logística (paramétrica, interpretable) vs. Árbol de Decisión (no paramétrico, captura interacciones no lineales). Comparamos con Accuracy y F1-macro.

In [ ]:
# ==========================================
# SESION 6 - REGRESION LOGISTICA VS CART
# ==========================================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

print("--- SESION 6: Modelos de Linea Base (Logistica vs CART) ---")

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
pred_log = log_model.predict(X_test_scaled)

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train_scaled, y_train)
pred_tree = tree_model.predict(X_test_scaled)

acc_log = accuracy_score(y_test, pred_log); f1_log = f1_score(y_test, pred_log, average="macro")
acc_tree = accuracy_score(y_test, pred_tree); f1_tree = f1_score(y_test, pred_tree, average="macro")

print(f"Regresion Logistica -> Accuracy: {acc_log:.4f} | F1-macro: {f1_log:.4f}")
print(f"Arbol CART           -> Accuracy: {acc_tree:.4f} | F1-macro: {f1_tree:.4f}")

**Ejercicio:** cambia `max_depth` del árbol a 2 y luego a 10. ¿Qué pasa con el F1-macro en cada caso? Relaciónalo con el concepto de sobreajuste (overfitting).

## Sesión 7 — Ensambles: Random Forest y XGBoost

Bagging (árboles en paralelo sobre muestras bootstrap) vs. Boosting (árboles secuenciales que corrigen el error residual). Validamos con 5-fold cross-validation y extraemos la importancia de variables.

In [ ]:
# ==========================================
# SESION 7 - ENSAMBLES (RANDOM FOREST Y XGBOOST)
# ==========================================
!pip install -q xgboost
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

print("--- SESION 7: Ensambles e Importancia de Variables ---")

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train_scaled, y_train)
pred_rf = rf_model.predict(X_test_scaled)

xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3,
                           random_state=42, eval_metric="mlogloss")
xgb_model.fit(X_train_scaled, y_train)
pred_xgb = xgb_model.predict(X_test_scaled)

cv_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring="f1_macro")
print(f"Random Forest (F1-macro promedio, 5-Fold CV): {cv_scores.mean():.4f}")

importancias = rf_model.feature_importances_
print("\nRanking de importancia de variables (Random Forest):")
for var, imp in sorted(zip(predictores, importancias), key=lambda x: x[1], reverse=True):
    print(f" -> {var:<24}: {imp*100:5.2f}%")

**Ejercicio:** anota los 3 sensores con mayor importancia en tu corrida. ¿Coinciden con `Soil_pH` y `Soil_Moisture`? Propón una explicación agronómica de por qué esos sensores dominan la predicción.

## Sesión 8 — Pipeline consolidado y evaluación final

Comparamos los 4 modelos, elegimos el mejor por F1-macro y generamos matriz de confusión y `classification_report` del ganador.

In [ ]:
# ==========================================
# SESION 8 - PIPELINE CONSOLIDADO Y EVALUACION FINAL
# ==========================================
from sklearn.metrics import confusion_matrix, classification_report

print("--- SESION 8: Evaluacion y Cierre del Proyecto ---")

acc_rf = accuracy_score(y_test, pred_rf); f1_rf = f1_score(y_test, pred_rf, average="macro")
acc_xgb = accuracy_score(y_test, pred_xgb); f1_xgb = f1_score(y_test, pred_xgb, average="macro")

print("\n=======================================================")
print("        TABLA DE RENDIMIENTO FINAL DE MODELOS          ")
print("=======================================================")
print(f" 1. Regresion Logistica  -> Accuracy: {acc_log:.4f} | F1-macro: {f1_log:.4f}")
print(f" 2. Arbol CART           -> Accuracy: {acc_tree:.4f} | F1-macro: {f1_tree:.4f}")
print(f" 3. Random Forest        -> Accuracy: {acc_rf:.4f} | F1-macro: {f1_rf:.4f}")
print(f" 4. XGBoost              -> Accuracy: {acc_xgb:.4f} | F1-macro: {f1_xgb:.4f}")
print("=======================================================")

resultados = {"Regresion Logistica": f1_log, "Arbol CART": f1_tree,
              "Random Forest": f1_rf, "XGBoost": f1_xgb}
mejor_modelo = max(resultados, key=resultados.get)
print(f"\n[EXITO] Modelo recomendado (mayor F1-macro): {mejor_modelo}")

print("\nMatriz de confusion - mejor modelo (Random Forest):")
print(confusion_matrix(y_test, pred_rf))
print("\nReporte de clasificacion (Random Forest):")
print(classification_report(y_test, pred_rf, target_names=le.classes_))

print("\nEl pipeline incremental de 8 sesiones se ejecuto correctamente de principio a fin.")

**Ejercicio final:** con tus propios resultados, escribe 3-4 líneas de conclusión: ¿qué modelo elegirías para desplegar en campo y por qué, considerando tanto el F1-macro como el recall de la clase "High Stress"?